# Табличные данные и визуализация

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи самостоятельно, затем обратитесь к разделу решений.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Выбор и группировка в pandas](../../python/numerical-data/pandas-selection/)
- [Объединения, пропуски и дубликаты](../../python/numerical-data/pandas-joins/)
- [Графики и исследование данных](../../python/numerical-data/visualization/)


## Чему научимся

Исследовать таблицу, проверять пропуски и объединения, выбирать график под вопрос. Нужны CSV и NumPy; установите pandas и matplotlib по введению.

## DataFrame и Series

`DataFrame` — таблица с именованными столбцами и индексом строк. Один столбец — `Series`. Индекс — метка строки, он не обязан совпадать с её позицией или быть уникальным. Создадим вымышленные измерения четырёх планеров. Подключим pandas под именем `pd`, а средства рисования matplotlib — под именем `plt`. `pd.DataFrame` строит таблицу из словаря: ключи задают столбцы, списки — их значения. Свойство `shape` задаёт число строк и столбцов.

`None` в числовом столбце становится пропуском; его конкретное представление зависит от типа столбца. `frame["mass_g"]` выбирает один столбец; его `isna()` даёт маску пропусков, а `sum()` считает значения `True` как единицы. Свойство `dtypes` показывает типы столбцов. Проверяйте типы (`dtypes`), первые строки (`head`) и сводку (`describe`) перед расчётами. Для чтения своего файла используйте `pd.read_csv(path)`; разделитель и кодировку задавайте по формату файла.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

frame = pd.DataFrame({
    "id": ["a", "b", "c", "d"],
    "span_cm": [12., 22., 14., 24.],
    "mass_g": [8., 10., None, 12.],
    "flight": ["короткий", "дальний", "короткий", "дальний"],
})
print(frame.shape)              # (4, 4)
print(frame["mass_g"].isna().sum())  # 1
print(frame.dtypes)


## Выбор и группировка

`.loc` выбирает по меткам или логической маске, `.iloc` — по позициям. `frame.loc[маска, список_столбцов]` оставляет нужные строки и столбцы, `frame.iloc[0]` возвращает первую строку. Метод `tolist()` превращает столбец в обычный список, `to_dict()` — результат с метками в словарь. `groupby` делит строки на группы перед вычислением показателя; `mean()` считает среднее, `count()` — число заполненных значений, `size()` — число всех строк группы. Для изменения используйте одно обращение `frame.loc[mask, "column"] = value`, а не цепочку промежуточных выборок. Если нужна независимая рабочая таблица, сделайте `.copy()`.

In [ ]:
selected = frame.loc[frame["span_cm"] >= 20, ["id", "span_cm"]]
assert selected["id"].tolist() == ["b", "d"]
assert frame.iloc[0]["id"] == "a"
means = frame.groupby("flight")["span_cm"].mean()
assert means["короткий"] == 13
assert means["дальний"] == 23


Среднее по умолчанию пропускает отсутствующие значения. Поэтому вместе с ним полезно считать количество заполненных (`count`) и всех строк (`size`). Две группы с одним и со ста измерениями дают оценки разной надёжности.

## Пропуски и дубликаты

`is_unique` истинно, если в столбце нет повторов. `duplicated()` отмечает повторяющиеся строки, начиная со второго вхождения; сумма маски даёт их число. `fillna(value)` возвращает столбец, в котором пропуски заменены на `value`.

In [ ]:
assert frame["id"].is_unique
assert frame.duplicated().sum() == 0
filled = frame.copy()
filled["mass_g"] = filled["mass_g"].fillna(10.)
assert filled["mass_g"].isna().sum() == 0
assert frame["mass_g"].isna().sum() == 1


Число 10 здесь — заранее выбранное учебное заполнение, не результат анализа проверочных данных. В модели медиану или другое правило подбирают на обучающей части. До удаления повторов выясните, что повторилось: вся строка, идентификатор или законное повторное измерение. Удаление одинаковых строк не гарантирует уникальности `id`.

## Разобранный пример: объединение с контролем

Добавим данные о бумаге. `merge` объединяет по ключу, `how="left"` сохраняет все строки левой таблицы. `validate="one_to_one"` требует уникальных ключей с обеих сторон; `indicator=True` добавляет столбец `_merge` со сведениями о совпадении ключа. Значение `"both"` означает, что ключ найден в обеих таблицах; `.all()` проверяет истинность условия для каждой строки. Поле `paper_gsm` — масса бумаги площадью один квадратный метр, в г/м².

In [ ]:
paper = pd.DataFrame({"id": ["a", "b", "c", "d"], "paper_gsm": [80, 90, 80, 90]})
joined = frame.merge(paper, on="id", how="left", validate="one_to_one", indicator=True)
assert len(joined) == len(frame)
assert (joined["_merge"] == "both").all()


Если справа два одинаковых ключа, строки могут размножиться. Если ключ отсутствует, появится пропуск. Проверяйте и число строк, и то, для каких ключей нашлось соответствие. Для справочника с повторяющимися ключами слева и уникальными справа подходит `many_to_one`. Не допускайте пустых ключей: их объединение может иметь неожиданный смысл.

## График отвечает на вопрос

Гистограмма показывает распределение одного числового признака по интервалам, диаграмма рассеяния — связь двух признаков, столбчатая диаграмма — сравнение категорий. Видимая связь на графике сама по себе не доказывает причинную связь: обе величины могут меняться из-за третьей. Одинаковые средние тоже не гарантируют похожего разброса: у наборов `[10, 10]` и `[1, 19]` одно среднее 10, но очень разные значения.

`plt.subplots(2, 1, figsize=(5, 6))` создаёт рисунок `fig` и две области графиков `axes` друг под другом; `figsize` задаёт размер в дюймах. Метод `hist` строит гистограмму, `bins` задаёт границы её интервалов, `scatter` — диаграмму рассеяния. Методы `set_xlabel` и `set_ylabel` подписывают оси, `set_xticks` и `set_yticks` задают положения делений. Для числа объектов выбираем целые деления. `tight_layout` подбирает отступы, `show` показывает рисунок, `close` освобождает его после показа.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5, 6))
axes[0].hist(frame["span_cm"], bins=[10, 15, 20, 25])
axes[0].set_xlabel("Размах, см")
axes[0].set_ylabel("Число объектов")
axes[0].set_xticks([10, 15, 20, 25])
axes[0].set_yticks([0, 1, 2])
axes[1].scatter(frame["span_cm"], frame["mass_g"])
axes[1].set_xlabel("Размах, см")
axes[1].set_ylabel("Масса, г")
fig.tight_layout()
plt.show()
plt.close(fig)


Ожидаем столбцы высотой 2, 0, 2. На диаграмме рассеяния будет три точки: строка с пропущенной массой не рисуется. Подписи с единицами позволяют понять график без чтения кода. Четырёх вымышленных объектов недостаточно для выводов о реальных планерах.

Для справки: [объединение таблиц в pandas](https://pandas.pydata.org/docs/user_guide/merging.html).

## Задачи

### 1. Базовая

Чем отличаются `.loc` и `.iloc`?

<details>
<summary>Подсказка</summary>

Метка и позиция могут быть разными.

</details>



### 2. Базовая

Сколько заполненных значений mass_g в примере?

<details>
<summary>Подсказка</summary>

Всего четыре строки, один пропуск.

</details>



### 3. Прикладная

Посчитайте число объектов каждого класса.

<details>
<summary>Подсказка</summary>

Метод `size` группы считает строки.

</details>



### 4. Прикладная

Найдите строки с пропуском массы и верните их id.

<details>
<summary>Подсказка</summary>

Постройте маску `isna`.

</details>



### 5. Олимпиадная

Слева ключ a повторён дважды, справа трижды. Сколько строк с ключом a получится при объединении?

<details>
<summary>Подсказка</summary>

Каждая строка слева сочетается с каждой справа.

</details>



### 6. Олимпиадная

У группы A средняя масса рассчитана по 100 строкам, у B по одной. Что скрывает график только двух средних?

<details>
<summary>Подсказка</summary>

Среднее не показывает число измерений и разброс.

</details>



## Итоги и самопроверка

Сформулируйте вопрос перед построением графика. После объединения проверьте ключи, число строк и пропуски; перед средним — типы и число наблюдений. Затем переходите к конвейерам моделей.


## Решения задач

### Задача 1

`.loc` обращается по меткам и маскам, `.iloc` — по целочисленным позициям.

### Задача 2

Три.

### Задача 3

In [ ]:
sizes = frame.groupby("flight").size()
assert sizes.to_dict() == {"дальний": 2, "короткий": 2}


### Задача 4

In [ ]:
missing_ids = frame.loc[frame["mass_g"].isna(), "id"].tolist()
assert missing_ids == ["c"]


### Задача 5

Шесть. Такой рост может незаметно изменить средние и доли; укажите ожидаемую связь через `validate`.

### Задача 6

Он скрывает объём данных, пропуски и вариативность. Нужно показать хотя бы числа наблюдений, а при достаточных данных — распределения и неопределённость. Сходные средние не означают одинаковую надёжность.